# Phase 1 — SROIE data preparation and EDA

This notebook downloads and inspects `darentang/sroie`, checks the OCR words, boxes, and BIO labels against the published schema, visualizes an annotated receipt, creates leakage-safe data splits, and saves prepared data to Google Drive.

## Create and configure Colab
1. Open [Google Colab](https://colab.research.google.com/) → **New notebook**.
2. In **Runtime → Change runtime type**, select a GPU if one is offered. GPU availability varies. This data-preparation phase can also run on CPU.
3. Upload this notebook or copy its cells into Colab; run cells in order.

The hosted dataset definition documents `id`, `words`, `bboxes`, `ner_tags`, and `image_path`, nine BIO tags, and official `train`/`test` splits. This notebook verifies these details after loading. [Published schema](https://huggingface.co/datasets/darentang/sroie/blob/main/sroie.py).

## Cell 1 — Runtime check and dependencies
Colab usually provides PyTorch. This cell reports Python/PyTorch/GPU information and installs only data-preparation dependencies.

In [ ]:
import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    import torch
    print("PyTorch:", torch.__version__)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    else:
        print("GPU unavailable — Phase 1 can run on CPU.")
except ImportError:
    print("PyTorch is absent; it is not required for Phase 1 EDA.")

%pip -q install "datasets>=3.0,<4.0" pandas matplotlib pillow scikit-learn pyarrow tqdm gdown

## Cell 2 — Mount Drive and create persistent folders
Prepared images are copied from the runtime dataset cache into Drive so they remain available after Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/intelligent-document-ai")
DATA_RAW = DRIVE_ROOT / "data" / "raw"
DATA_PROCESSED = DRIVE_ROOT / "data" / "processed"
ANNOTATIONS = DRIVE_ROOT / "data" / "annotations"
for folder in [DATA_RAW, DATA_PROCESSED, ANNOTATIONS,
               DRIVE_ROOT / "checkpoints", DRIVE_ROOT / "models",
               DRIVE_ROOT / "metrics", DRIVE_ROOT / "outputs"]:
    folder.mkdir(parents=True, exist_ok=True)
print("Persistent project folder:", DRIVE_ROOT)

## Cell 3 — Download and inspect actual splits and features
The repository uses a dataset loading script, so the compatible `datasets` release is pinned below 4. If the upstream loader changes, this cell reports the original error instead of guessing a schema.

In [ ]:
from datasets import load_dataset

DATASET_ID = "darentang/sroie"
try:
    raw = load_dataset(DATASET_ID, trust_remote_code=True)
except Exception as exc:
    raise RuntimeError(
        "Could not load darentang/sroie. Check the current dataset page and datasets compatibility. "
        "Do not substitute guessed columns or labels. Original error: " + repr(exc)
    ) from exc

print("Splits:", list(raw.keys()))
for split_name, split_data in raw.items():
    print(f"\n{split_name}: {len(split_data):,} rows")
    print(split_data.features)

expected_columns = {"id", "words", "bboxes", "ner_tags", "image_path"}
if not {"train", "test"}.issubset(raw.keys()):
    raise ValueError(f"Expected official train and test; found {list(raw.keys())}")
for split_name in ("train", "test"):
    actual = set(raw[split_name].column_names)
    if not expected_columns.issubset(actual):
        raise ValueError(f"{split_name} schema mismatch: expected {expected_columns}, got {actual}")
label_feature = raw["train"].features["ner_tags"].feature
label_names = label_feature.names
print("BIO labels:", label_names)

## Cell 4 — Validate annotations and inspect labels
Each OCR word should align with exactly one box and one tag. The published loader normalizes boxes to 0–1000 coordinates. We report invalid examples and stop instead of silently modifying them.

In [ ]:
from collections import Counter

def tag_name(tag, feature):
    # ClassLabel values may be returned as strings or integer IDs by library versions.
    return tag if isinstance(tag, str) else feature.int2str(int(tag))

def inspect_split(data, split_name, feature):
    problems, counts = [], Counter()
    for row_index, row in enumerate(data):
        words, boxes, tags = row["words"], row["bboxes"], row["ner_tags"]
        if not (len(words) == len(boxes) == len(tags)):
            problems.append((row_index, row.get("id"), "word/box/tag lengths differ"))
            continue
        for word_index, (word, box, tag) in enumerate(zip(words, boxes, tags)):
            name = tag_name(tag, feature)
            counts[name] += 1
            if not isinstance(word, str) or not word.strip():
                problems.append((row_index, row.get("id"), f"empty word at {word_index}"))
            if len(box) != 4 or not all(isinstance(v, (int, float)) for v in box):
                problems.append((row_index, row.get("id"), f"invalid box at {word_index}: {box}"))
            elif not (0 <= box[0] <= box[2] <= 1000 and 0 <= box[1] <= box[3] <= 1000):
                problems.append((row_index, row.get("id"), f"box outside 0–1000 at {word_index}: {box}"))
            if name not in feature.names:
                problems.append((row_index, row.get("id"), f"unknown tag: {name}"))
    print(f"{split_name}: {len(data)} docs; {sum(counts.values()):,} words")
    print("Tag counts:", dict(counts))
    print("Problem count:", len(problems))
    if problems:
        print("First problems:", problems[:10])
    return problems, counts

train_problems, train_tag_counts = inspect_split(raw["train"], "train", label_feature)
test_problems, test_tag_counts = inspect_split(raw["test"], "test", label_feature)
if train_problems or test_problems:
    raise ValueError("Dataset validation found invalid records; review the examples above.")

sample = raw["train"][0]
print("Example ID:", sample["id"])
for word, tag in list(zip(sample["words"], sample["ner_tags"]))[:25]:
    print(f"{word!r:24} {tag_name(tag, label_feature)}")

## Cell 5 — Visualize one receipt and its word boxes
Red boxes show the provided word annotations. Inspect the image to confirm the normalized coordinates align with the receipt.

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Some hosted parquet rows contain the dataset builder's cache path from the
# Hub server (for example, /storage/hf-datasets-cache/...). That path is not
# available in Colab. If needed, recover the official images archive once.
import zipfile
import gdown

SOURCE_ARCHIVE_ID = "1ZyxAw1d-9UvhgNLGRvsJK4gBCMf0VpGD"  # referenced by the dataset's published loader
SOURCE_ARCHIVE = Path("/content/sroie_source.zip")
SOURCE_EXTRACTED = Path("/content/sroie_source")

def resolve_image_path(row, source_split):
    candidate = Path(row["image_path"])
    if candidate.is_file():
        return candidate

    if not SOURCE_EXTRACTED.exists():
        if not SOURCE_ARCHIVE.exists():
            result = gdown.download(id=SOURCE_ARCHIVE_ID, output=str(SOURCE_ARCHIVE), quiet=False)
            if result is None or not SOURCE_ARCHIVE.is_file():
                raise RuntimeError(
                    "The dataset row points to a non-local image and the original SROIE archive "
                    "could not be downloaded from Google Drive. Check access/quota for the dataset's source archive."
                )
        if not zipfile.is_zipfile(SOURCE_ARCHIVE):
            raise RuntimeError(f"Downloaded SROIE source is not a ZIP archive: {SOURCE_ARCHIVE}")
        SOURCE_EXTRACTED.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
            archive.extractall(SOURCE_EXTRACTED)

    matches = [path for path in SOURCE_EXTRACTED.rglob(candidate.name)
               if path.is_file() and path.parent.name.lower() == "images"
               and source_split.lower() in {part.lower() for part in path.parts}]
    if not matches:
        raise FileNotFoundError(
            f"Could not find {candidate.name} in the downloaded SROIE {source_split} image folder. "
            "Inspect the archive structure and current dataset source before changing annotations."
        )
    return matches[0]

SAMPLE_INDEX = 0
sample = raw["train"][SAMPLE_INDEX]
image_path = resolve_image_path(sample, "train")
image = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image)
width, height = image.size
for box in sample["bboxes"]:
    x0, y0, x1, y1 = box
    pixel_box = (round(x0 * width / 1000), round(y0 * height / 1000),
                 round(x1 * width / 1000), round(y1 * height / 1000))
    draw.rectangle(pixel_box, outline="red", width=2)
plt.figure(figsize=(10, 12))
plt.imshow(image)
plt.title(f"{sample['id']} — red boxes are annotation boxes")
plt.axis("off");

## Cell 6 — Split documents without leakage
The official test split is preserved. Validation comes only from the official training split. A fixed seed makes the document-level split reproducible.

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

SEED = 42
VALIDATION_FRACTION = 0.10
train_indices, val_indices = train_test_split(
    np.arange(len(raw["train"])),
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    shuffle=True,
)
split_rows = {
    "train": [raw["train"][int(i)] for i in train_indices],
    "validation": [raw["train"][int(i)] for i in val_indices],
    "test": [row for row in raw["test"]],
}
# Include the original source split in the key because IDs can be generated per split.
train_ids = {("train", str(row["id"])) for row in split_rows["train"]}
val_ids = {("train", str(row["id"])) for row in split_rows["validation"]}
assert train_ids.isdisjoint(val_ids)
print({name: len(rows) for name, rows in split_rows.items()})
print("Train/validation overlap:", len(train_ids & val_ids))
print("Official test documents retained:", len(split_rows["test"]))

## Cell 7 — Save prepared JSONL and image files to Drive
The hosted data already has normalized boxes; this cell preserves them (no second normalization). Each record stores words, boxes, tag names, document ID, split, and a persistent image path.

In [ ]:
import json, shutil
from tqdm.auto import tqdm

RAW_IMAGES = DATA_RAW / "images"
for split_name, rows in split_rows.items():
    image_folder = RAW_IMAGES / split_name
    image_folder.mkdir(parents=True, exist_ok=True)
    source_split = "test" if split_name == "test" else "train"
    output_path = DATA_PROCESSED / f"{split_name}.jsonl"
    with output_path.open("w", encoding="utf-8") as out_file:
        for row in tqdm(rows, desc=f"Saving {split_name}"):
            source_image = resolve_image_path(row, source_split)
            if not source_image.exists():
                raise FileNotFoundError(f"Missing image for {row['id']}: {source_image}")
            suffix = source_image.suffix.lower() or ".jpg"
            destination = image_folder / f"{row['id']}{suffix}"
            if not destination.exists():
                shutil.copy2(source_image, destination)
            words = list(row["words"])
            boxes = [[int(v) for v in box] for box in row["bboxes"]]
            tags = [tag_name(tag, label_feature) for tag in row["ner_tags"]]
            if not (len(words) == len(boxes) == len(tags)):
                raise ValueError(f"Unaligned fields in {split_name}/{row['id']}")
            record = {
                "document_id": str(row["id"]), "split": split_name,
                "image_path": str(destination), "words": words,
                "bboxes": boxes, "ner_tags": tags,
            }
            out_file.write(json.dumps(record, ensure_ascii=False) + "\n")

metadata = {
    "dataset": DATASET_ID,
    "seed": SEED,
    "validation_fraction_from_official_train": VALIDATION_FRACTION,
    "source_splits": list(raw.keys()),
    "label_names": label_names,
    "document_counts": {name: len(rows) for name, rows in split_rows.items()},
}
(ANNOTATIONS / "phase1_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Prepared records:", DATA_PROCESSED)
print("Metadata:", ANNOTATIONS / "phase1_metadata.json")

## Cell 8 — Validate every saved output
Checks JSONL readability, word/box/tag alignment, normalized box ranges, known labels, and image existence. These are data integrity checks, not model-performance results.

In [ ]:
for split_name in ("train", "validation", "test"):
    jsonl_path = DATA_PROCESSED / f"{split_name}.jsonl"
    checked = 0
    with jsonl_path.open(encoding="utf-8") as file:
        for line in file:
            row = json.loads(line)
            assert len(row["words"]) == len(row["bboxes"]) == len(row["ner_tags"])
            assert all(0 <= b[0] <= b[2] <= 1000 and 0 <= b[1] <= b[3] <= 1000
                       for b in row["bboxes"])
            assert Path(row["image_path"]).is_file()
            assert set(row["ner_tags"]).issubset(set(label_names))
            checked += 1
    print(f"{split_name}: validated {checked} records")
print("Phase 1 output checks completed in this runtime.")

## Expected outputs and recovery
Drive contains `data/processed/{train,validation,test}.jsonl`, copied images under `data/raw/images/{train,validation,test}/`, and `data/annotations/phase1_metadata.json`. No model metrics are produced here.

After a Colab disconnect, reconnect and rerun the runtime check, dependency, Drive, and data cells. Prepared Drive files persist; a new runtime may need to download the source dataset again. Review the displayed receipt and validation output before continuing. If a schema or annotation check fails, stop and inspect the live dataset rather than guessing a correction.